# Dash-cam fine-tune of the pothole model (YOLO11n) — Colab T4

The first model (`train_pothole.ipynb`, mAP50 0.85) learned mostly from close-up photos, so on
real dash-cam frames it **misses far-away potholes**. This notebook fixes that:

- **Dash-cam data only:** Figshare "Potholes dataset with YOLO annotations" (2000+ Full HD dash-cam
  frames) + Roboflow `potholes-ptvvw` v2 `dashcam_832`. Close-up photos are left out.
- **Starts from your first `best.pt`** (already knows potholes) → learns the new angle quickly.
- **`imgsz=960`** instead of 640 → far potholes keep ~2× more pixels.
- **Honest test split:** Figshare frames are split in file-name order (whole stretches of video go
  to test), so near-identical neighbouring frames can't leak between train and test.
- **Old vs new scored on the same held-out frames**, then both ONNX sizes exported.

**Before you start:** same as the first notebook — T4 GPU, 🔑 `ROBOFLOW_API_KEY` in Secrets, and the
first notebook must have finished (it reads `still-there-potholes/handoff/best.pt`).
`Runtime → Run all`. ~45–60 min. Saves to Drive; re-run all to resume after a disconnect.

## 1 · GPU + Drive + install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU — Runtime > Change runtime type > T4 GPU"
!pip -q install ultralytics roboflow
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/still-there-potholes'
OLD_BEST = f'{DRIVE_DIR}/handoff/best.pt'
assert os.path.exists(OLD_BEST), 'Run train_pothole.ipynb first — it makes handoff/best.pt'
print('starting from', OLD_BEST)

## 2 · Download the two dash-cam datasets

In [ ]:
import glob, zipfile, requests, getpass
RAW = '/content/datasets'

# Figshare dash-cam set (public API, no key)
FS = f'{RAW}/figshare_dashcam'; os.makedirs(FS, exist_ok=True)
meta = requests.get('https://api.figshare.com/v2/articles/21214400', timeout=30).json()
assert meta.get('files'), f'Figshare returned no files: {str(meta)[:300]}'
for f in meta['files']:
    dst = f"{FS}/{f['name']}"
    if os.path.exists(dst) and os.path.getsize(dst) == f['size']:
        print('have', f['name']); continue
    print(f"downloading {f['name']} ({f['size']/1e6:.0f} MB)...")
    with requests.get(f['download_url'], stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dst, 'wb') as out:
            for chunk in r.iter_content(1 << 20): out.write(chunk)
seen = set()
while True:
    zips = [z for z in glob.glob(f'{FS}/**/*.zip', recursive=True) if z not in seen]
    if not zips: break
    for z in zips:
        seen.add(z); print('unzipping', os.path.basename(z)); zipfile.ZipFile(z).extractall(os.path.splitext(z)[0])

# Roboflow dash-cam set (same one as the first notebook)
try:
    from google.colab import userdata; RF_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    RF_KEY = None
if isinstance(RF_KEY, dict): RF_KEY = RF_KEY.get('value') or next((v for v in RF_KEY.values() if isinstance(v, str)), None)
RF_KEY = RF_KEY if isinstance(RF_KEY, str) and RF_KEY else getpass.getpass('Roboflow API key: ')
from roboflow import Roboflow
RFD = f'{RAW}/dashcam'
if not os.path.exists(f'{RFD}/data.yaml'):
    v = Roboflow(api_key=RF_KEY).workspace('work-5qhak').project('potholes-ptvvw').version(2)
    try: v.download('yolov11', location=RFD)
    except Exception as e: print('yolov11 failed, using yolov8:', e); v.download('yolov8', location=RFD)
print('downloads done')

## 3 · Build one dash-cam dataset (one class `pothole`)

Figshare: images sorted by file name and cut into contiguous blocks — first 80% train, next 10% valid,
last 10% test — so frames from the same stretch of road stay together. Images whose label file is
**empty** are kept as "no pothole here" backgrounds (they teach it what *not* to box).
Roboflow dash-cam: its own train/valid/test splits, every class → 0.

In [ ]:
BUILD_CODE = True  # marker for the offline test
import os, glob, shutil, yaml
from collections import Counter

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')
OUT = '/content/dash'

def to_box_rows(label_path):
    rows = []
    if not label_path or not os.path.exists(label_path):
        return rows
    for line in open(label_path):
        p = line.split()
        if len(p) < 5: continue
        nums = [float(x) for x in p[1:]]
        if len(nums) > 4:                                    # polygon -> box
            xs, ys = nums[0::2], nums[1::2]
            nums = [(min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)]
        rows.append('0 ' + ' '.join(f'{v:.6f}' for v in nums[:4]))
    return rows

def put(out, split, img, rows, name, stats):
    for d in ('images', 'labels'): os.makedirs(f'{out}/{split}/{d}', exist_ok=True)
    shutil.copy(img, f'{out}/{split}/images/{name}{os.path.splitext(img)[1].lower()}')
    with open(f'{out}/{split}/labels/{name}.txt', 'w') as f:
        f.write('\n'.join(rows) + ('\n' if rows else ''))
    stats[split] += 1; stats[split + '_boxes'] += len(rows); stats[split + '_background'] += (not rows)

def add_figshare(src, out, stats, frac=(0.8, 0.1)):
    labels = {os.path.splitext(os.path.basename(p))[0]: p
              for p in glob.glob(f'{src}/**/*.txt', recursive=True) if 'classes' not in os.path.basename(p).lower()}
    imgs = sorted(p for p in glob.glob(f'{src}/**/*', recursive=True)
                  if p.lower().endswith(IMG_EXT) and os.path.splitext(os.path.basename(p))[0] in labels)
    n = len(imgs); a, b = int(n * frac[0]), int(n * (frac[0] + frac[1]))
    for i, img in enumerate(imgs):
        split = 'train' if i < a else 'valid' if i < b else 'test'
        stem = os.path.splitext(os.path.basename(img))[0]
        put(out, split, img, to_box_rows(labels[stem]), f'fs_{i:05d}', stats)
    print(f'figshare: {n} labeled images (unlabeled skipped)')

def add_roboflow(src, out, stats, tag):
    for split in ('train', 'valid', 'test'):
        d = next((f'{src}/{c}' for c in (split, 'val') if os.path.isdir(f'{src}/{c}/images')), None)
        if not d: continue
        for img in glob.glob(f'{d}/images/*'):
            stem = os.path.splitext(os.path.basename(img))[0]
            put(out, split, img, to_box_rows(f'{d}/labels/{stem}.txt'), f'{tag}_{stem}', stats)

def build(fs_src, rf_src, out=OUT):
    shutil.rmtree(out, ignore_errors=True)
    stats = Counter()
    add_figshare(fs_src, out, stats)
    test_only = f'{out}_fstest'                              # Figshare test split alone, for old-vs-new
    shutil.rmtree(test_only, ignore_errors=True)
    shutil.copytree(f'{out}/test', f'{test_only}/test')
    add_roboflow(rf_src, out, stats, 'rf')
    for root in (out, test_only):
        cfg = {'path': root, 'train': ('train/images' if root == out else 'test/images'),
               'val': ('valid/images' if root == out else 'test/images'), 'test': 'test/images',
               'names': {0: 'pothole'}}
        yaml.safe_dump(cfg, open(f'{root}/data.yaml', 'w'))
    print(dict(stats))
    return f'{out}/data.yaml', f'{test_only}/data.yaml'

if 'FS' in globals():
    DATA_YAML, FSTEST_YAML = build(FS, RFD)

## 4 · Baseline: how the OLD model does on the held-out dash-cam frames

In [ ]:
from ultralytics import YOLO
def score(weights, data, imgsz):
    m = YOLO(weights).val(data=data, split='test', imgsz=imgsz, verbose=False, plots=False)
    return m.box.map50, m.box.mp, m.box.mr
results = {}
for sz in (640, 960):
    results[f'old @ {sz}'] = score(OLD_BEST, FSTEST_YAML, sz)
    print(f'old model @ imgsz {sz}:  mAP50={results[f"old @ {sz}"][0]:.3f}  P={results[f"old @ {sz}"][1]:.3f}  R={results[f"old @ {sz}"][2]:.3f}')

## 5 · Fine-tune at 960 (≈10–20 min on an A100, ≈35–50 min on a T4)

Starts from the old `best.pt`. Saves every epoch to Drive; re-run all to resume.

In [ ]:
RUN = f'{DRIVE_DIR}/runs/y11n_dashcam960'
last, done = f'{RUN}/weights/last.pt', f'{RUN}/weights/TRAINING_DONE'
if os.path.exists(done):
    print('Already trained — skipping. Delete', done, 'to train again.')
elif os.path.exists(last):
    print('Resuming', last); YOLO(last).train(resume=True)
else:
    YOLO(OLD_BEST).train(
        data=DATA_YAML, epochs=30, imgsz=960, batch=-1, patience=10,   # fine-tune: 30 is plenty
        close_mosaic=5, cache='ram', workers=min(8, os.cpu_count() or 2),   # A100 boxes have more CPUs
        project=f'{DRIVE_DIR}/runs', name='y11n_dashcam960', exist_ok=True,
    )
open(done, 'w').close()
NEW_BEST = f'{RUN}/weights/best.pt'

## 6 · Old vs new on the same held-out dash-cam frames

In [ ]:
for sz in (640, 960):
    results[f'new @ {sz}'] = score(NEW_BEST, FSTEST_YAML, sz)
print(f"{'model':12s} {'mAP50':>7s} {'P':>7s} {'R (found)':>10s}")
for k in ('old @ 640', 'old @ 960', 'new @ 640', 'new @ 960'):
    m, p, r = results[k]; print(f'{k:12s} {m:7.3f} {p:7.3f} {r:10.3f}')
print('\nnew @ 640 = what the phone page gets;  new @ 960 = what the drive pipeline gets')

## 7 · Look at it: old vs new on the same 4 held-out frames

In [ ]:
import random, matplotlib.pyplot as plt
frames = random.sample(glob.glob('/content/dash_fstest/test/images/*'), 4)
old_r = YOLO(OLD_BEST).predict(frames, conf=0.3, imgsz=640, verbose=False)
new_r = YOLO(NEW_BEST).predict(frames, conf=0.3, imgsz=960, verbose=False)
fig, axes = plt.subplots(4, 2, figsize=(18, 20))
for row, (o, n) in enumerate(zip(old_r, new_r)):
    for col, (r, title) in enumerate(((o, 'OLD @640'), (n, 'NEW @960'))):
        axes[row, col].imshow(r.plot()[..., ::-1]); axes[row, col].axis('off')
        axes[row, col].set_title(f'{title}: {len(r.boxes)} pothole(s)')
plt.tight_layout(); plt.show()

## 8 · Export + hand off

- `pothole.onnx` — 640 input, drop-in replacement for Ado's phone page (same file name, same format).
- `pothole_960.onnx` — 960 input, for the drive pipeline (or for the phone if WebGPU is fast enough;
  Ado would set the page's input size to 960).
- `best.pt` — for the Python pipeline.

Saved to `still-there-potholes/handoff_dashcam/`. **Only replace Ado's model if step 6 shows
`new @ 640` beating `old @ 640`.**

In [ ]:
import shutil
out = f'{DRIVE_DIR}/handoff_dashcam'; os.makedirs(out, exist_ok=True)
for sz, name in ((640, 'pothole.onnx'), (960, 'pothole_960.onnx')):
    f = YOLO(NEW_BEST).export(format='onnx', imgsz=sz)
    shutil.copy(f, f'{out}/{name}')
shutil.copy(NEW_BEST, f'{out}/best.pt')
for f in sorted(os.listdir(out)):
    print(f'{f:18s} {os.path.getsize(f"{out}/{f}")/1e6:.1f} MB')
print('\nsaved to', out)